In [1]:
%pip install transformers bitsandbytes accelerate torch

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from larg

In [1]:
import os
print(os.getpid())

1096


In [3]:
import os

folder_path = "tables/"

folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"

folder_path_python_code = "LLAMA/c.checking_statements"

folder_path_python_output_checking_statements = "LLAMA/d.checking_statements_output"

Initialize model

In [4]:
#initalizing the model with 4 bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.1-70B-Instruct"

# Configure 4-bit quantization to save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [5]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [6]:
import re

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
csv_files.sort()  # optional: ensure consistent order

for csv_file in csv_files:
    full_path = os.path.join(folder_path, csv_file)

    # Read the table
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{csv_file} is empty, skipping")
        continue

    # Extract header and rows
    header = lines[0]
    rows = lines[1:]

    print(f"Processing {csv_file}: {len(rows)} rows")

    prompt1 = [
    {"role": "system", "content": "You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables."},
    {"role": "user", "content": f"""
Your task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.

REQUIREMENTS:
1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.
2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). Avoid arbitrary correlations.
3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions.
4. **Variety**: Mix different types of statements:
   - Universal claims: "All X satisfy property Y"
   - Existential claims: "There exists at least one X that satisfies Y"
   - Majority/frequency claims: "Most X have property Y"
   - Conditional claims: "If X, then Y"

AVOID:
- Overly specific conditions that apply to only 1-2 individuals
- Redundant statements that express the same fact in different ways
- Statements too complex to parse (keep conditions to 2-3 attributes max)
- Probabilistic language (e.g., "likely", "probably") unless you have strong statistical support

Here is the data:
{lines}

Generate your statements below, one per line."""},]
    
    #Generate the statements necessary
    generation = generator(
    prompt1,
    do_sample=False,
    temperature=1.0,
    top_p=1,
    max_new_tokens=10000,
    eos_token_id=tokenizer.eos_token_id)
    print(f"Generation: {generation[0]['generated_text']}")
    # Get the assistant message from generated_text
    statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
    clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
    statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
    # Preview
    print(statements_LLM_output_text[1000:2000])
    #save the output of the LLM generated tasks
    file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"

    folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"

    full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)


    with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
      f.write(statements_LLM_output_text)
    print(f"Saved {file_name_LLM}.")


Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing table_0.csv: 15 rows


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nYour task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). Avoid arbitrary correlations.\n3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions.\n4. **Variety**: Mix different types of statements:\n   - Universal claims: "All X satisfy property Y"\n   - Existential claims: "There exists at least one X that satisfies Y"\n   - Majority/frequency cl

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nYour task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). Avoid arbitrary correlations.\n3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions.\n4. **Variety**: Mix different types of statements:\n   - Universal claims: "All X satisfy property Y"\n   - Existential claims: "There exists at least one X that satisfies Y"\n   - Majority/frequency cl

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nYour task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). Avoid arbitrary correlations.\n3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions.\n4. **Variety**: Mix different types of statements:\n   - Universal claims: "All X satisfy property Y"\n   - Existential claims: "There exists at least one X that satisfies Y"\n   - Majority/frequency cl

In [1]:
import re
import subprocess
import pandas as pd

# helper to extract
def extract_python_code(raw: str) -> str:
    # Match ```python ... ``` or ``` ... ``` (non-greedy, dotall)
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    # No fences found — return as-is (model already output plain code)
    return raw.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    for csv_file in csv_files:
        if csv_file[:-4] not in LLM_statement_text_file:
            continue

        full_csv_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements {lines} and the header names of the table {df.head(0)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""},
        ]

        # ── Generate ───────────────────────────────────────────────────────────
        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]

        # ── Clean: strip markdown fences reliably ──────────────────────────────
        python_code = extract_python_code(raw_code)

        # ── Save ───────────────────────────────────────────────────────────────
        python_file_name_LLM = f"python_code_{csv_file[:-4]}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        # ── Execute automatically ──────────────────────────────────────────────
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_llama_inferences_{csv_file[:-4]}.txt"
        full_path_results_file = os.path.join(folder_path_python_output_checking_statements, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")

NameError: name 'folder_path_LLM_statements' is not defined